# Statistical Analysis v2 — Multi-Seed Zero-Shot LLM Classification
**Paper:** *From Paid APIs to Local GPUs: Zero-Shot Consumer Financial Complaint Classification with Open-Source LLMs*  
**Authors:** Muhammet Tan, Halit Bakır — Sivas University of Science and Technology

This notebook reproduces all numerical results for the **revised multi-seed experiments** (3 seeds × 2 sample sizes × 2 conditions × 8 models = 96 runs).  
Running every cell top-to-bottom regenerates every table and statistical test directly from the raw prediction CSV files in `preds/`.

**Requirements:** `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`

In [1]:
import glob, os, re
import numpy as np
import pandas as pd
from scipy.stats import binom
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report,
)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. Constants & Helpers

In [2]:
CATEGORIES = [
    "retail_banking", "credit_reporting", "credit_card",
    "mortgages_and_loans", "debt_collection",
]
REASONING = {"deepseek-r1_14b", "deepseek-r1_8b", "qwen3_8b", "gpt-oss_20b"}
MODELS = [
    "deepseek-r1_8b", "deepseek-r1_14b", "gemma2_9b", "gpt-oss_20b",
    "llama3.1_8b", "mistral_7b", "phi4_14b", "qwen3_8b",
]
MODEL_PAT = "|".join(re.escape(m) for m in MODELS)
FNAME_PAT = re.compile(
    rf"preds_({MODEL_PAT})_(\d+)_subset_(balanced|imbalanced)_(\d+)\.csv"
)
PREDS_DIR = "preds"

REFERENCE = {
    "o1 (paid API)":                {"acc": 0.777, "f1": 0.7764, "cost": "$24.24"},
    "GPT-4.5 (paid API)":           {"acc": 0.774, "f1": 0.7763, "cost": "$12.32"},
    "Claude 3.7 Sonnet (paid API)": {"acc": 0.765, "f1": 0.7626, "cost": "$0.64"},
    "Gemini 2.0 Flash (paid API)":  {"acc": 0.758, "f1": 0.7586, "cost": "$0.04"},
    "GPT-4o (paid API)":            {"acc": 0.736, "f1": 0.7358, "cost": "$0.45"},
}

def load(path):
    df = pd.read_csv(path)
    df["true_label"]      = df["true_label"].str.lower().str.strip()
    df["predicted_label"] = df["predicted_label"].str.lower().str.strip()
    df.loc[~df["predicted_label"].isin(CATEGORIES), "predicted_label"] = "off_label"
    return df

def compute_metrics(df):
    y_true, y_pred = df["true_label"], df["predicted_label"]
    per_class = f1_score(y_true, y_pred, labels=CATEGORIES, average=None, zero_division=0)
    lat = df["latency_seconds"].astype(float)
    return {
        "accuracy":       accuracy_score(y_true, y_pred),
        "precision":      precision_score(y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0),
        "recall":         recall_score(   y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0),
        "f1_weighted":    f1_score(       y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0),
        "f1_macro":       f1_score(       y_true, y_pred, labels=CATEGORIES, average="macro",    zero_division=0),
        "mean_latency_s": lat.mean(),
        "off_label":      int((y_pred == "off_label").sum()),
        "n":              len(df),
        **{f"f1_{c}": v for c, v in zip(CATEGORIES, per_class)},
    }

def load_all():
    records = []
    for path in sorted(glob.glob(os.path.join(PREDS_DIR, "*.csv"))):
        m = FNAME_PAT.match(os.path.basename(path))
        if not m:
            continue
        model, seed, condition, size = m.groups()
        df = load(path)
        metrics = compute_metrics(df)
        records.append({"model": model, "seed": int(seed), "condition": condition,
                         "size": int(size), "_df": df, **metrics})
    return pd.DataFrame(records)

METRIC_COLS = ["accuracy", "f1_weighted", "f1_macro", "precision", "recall",
               "mean_latency_s", "off_label"] + [f"f1_{c}" for c in CATEGORIES]

def aggregate(raw, condition, size):
    sub = raw[(raw["condition"] == condition) & (raw["size"] == size)]
    rows = []
    for model in MODELS:
        g = sub[sub["model"] == model]
        if g.empty: continue
        row = {"model": model, "n_seeds": len(g),
               "type": "Reasoning" if model in REASONING else "Standard"}
        for col in METRIC_COLS:
            vals = g[col].values.astype(float)
            row[f"{col}_mean"] = vals.mean()
            row[f"{col}_std"]  = vals.std(ddof=1) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows).sort_values("accuracy_mean", ascending=False).reset_index(drop=True)

print("Helpers loaded. Loading prediction files...")
raw = load_all()
print(f"  {len(raw)} files | models: {sorted(raw['model'].unique())}")
print(f"  seeds: {sorted(raw['seed'].unique())} | sizes: {sorted(raw['size'].unique())}")

Helpers loaded. Loading prediction files...


  96 files | models: ['deepseek-r1_14b', 'deepseek-r1_8b', 'gemma2_9b', 'gpt-oss_20b', 'llama3.1_8b', 'mistral_7b', 'phi4_14b', 'qwen3_8b']
  seeds: [np.int64(1), np.int64(2), np.int64(3)] | sizes: [np.int64(250), np.int64(500)]


---
## 2. Experiment A — Balanced (N=500, mean ± std over 3 seeds)

In [3]:
agg_a = aggregate(raw, "balanced", 500)

disp = agg_a[["model","type",
              "accuracy_mean","accuracy_std",
              "f1_weighted_mean","f1_weighted_std",
              "f1_macro_mean","f1_macro_std",
              "mean_latency_s_mean","off_label_mean"]].copy()
disp.columns = ["Model","Type","Acc","Acc±","F1w","F1w±","F1m","F1m±","Lat(s)","Off-label"]
for c in ["Acc","Acc±","F1w","F1w±","F1m","F1m±","Lat(s)"]:
    disp[c] = disp[c].round(4)
disp["Off-label"] = disp["Off-label"].round(1)

print("Experiment A — Balanced (N=500)")
print("=" * 95)
print(disp.to_string(index=False))

Experiment A — Balanced (N=500)
          Model      Type    Acc   Acc±    F1w   F1w±    F1m   F1m±  Lat(s)  Off-label
    gpt-oss_20b Reasoning 0.7483 0.0199 0.7494 0.0185 0.7494 0.0185  4.1295     0.0000
deepseek-r1_14b Reasoning 0.7237 0.0064 0.7240 0.0068 0.7240 0.0068  9.3279     1.3000
       qwen3_8b Reasoning 0.7167 0.0032 0.7172 0.0023 0.7172 0.0023  5.1940     0.0000
       phi4_14b  Standard 0.7050 0.0115 0.7056 0.0107 0.7056 0.0107  0.5444     1.0000
 deepseek-r1_8b Reasoning 0.6843 0.0129 0.6829 0.0119 0.6829 0.0119  5.1434     2.0000
      gemma2_9b  Standard 0.6797 0.0064 0.6763 0.0085 0.6763 0.0085  0.6749     0.3000
     mistral_7b  Standard 0.5933 0.0023 0.5868 0.0039 0.5868 0.0039  0.2345    58.7000
    llama3.1_8b  Standard 0.5896 0.0188 0.5982 0.0178 0.5974 0.0164  5.7751    56.0000


---
## 3. Experiment B — Imbalanced (N=500, mean ± std over 3 seeds)

In [4]:
agg_b = aggregate(raw, "imbalanced", 500)

disp_b = agg_b[["model","type",
                "accuracy_mean","accuracy_std",
                "f1_weighted_mean","f1_weighted_std",
                "f1_macro_mean","f1_macro_std",
                "mean_latency_s_mean","off_label_mean"]].copy()
disp_b.columns = ["Model","Type","Acc","Acc±","F1w","F1w±","F1m","F1m±","Lat(s)","Off-label"]
for c in ["Acc","Acc±","F1w","F1w±","F1m","F1m±","Lat(s)"]:
    disp_b[c] = disp_b[c].round(4)
disp_b["Off-label"] = disp_b["Off-label"].round(1)

print("Experiment B — Imbalanced (N=500)")
print("=" * 95)
print(disp_b.to_string(index=False))

# Class distribution
sample_df = raw[(raw["condition"]=="imbalanced") & (raw["seed"]==1) & (raw["size"]==500) & (raw["model"]=="gpt-oss_20b")]["_df"].values[0]
dist = sample_df["true_label"].value_counts(normalize=True).round(4) * 100
print("\nTrue class distribution (Experiment B):")
print(dist.to_string())

Experiment B — Imbalanced (N=500)
          Model      Type    Acc   Acc±    F1w   F1w±    F1m   F1m±  Lat(s)  Off-label
    gpt-oss_20b Reasoning 0.7820 0.0171 0.7825 0.0162 0.7402 0.0259  4.0105     0.0000
       qwen3_8b Reasoning 0.7663 0.0081 0.7621 0.0065 0.7191 0.0091  6.9681     0.3000
       phi4_14b  Standard 0.7572 0.0091 0.7573 0.0061 0.7089 0.0199  0.5783     0.0000
deepseek-r1_14b Reasoning 0.7478 0.0188 0.7494 0.0169 0.7059 0.0281  8.1664     0.3000
 deepseek-r1_8b Reasoning 0.7380 0.0121 0.7312 0.0112 0.6768 0.0166  4.4623     2.3000
      gemma2_9b  Standard 0.7200 0.0115 0.7193 0.0139 0.6558 0.0150  0.6668     0.3000
     mistral_7b  Standard 0.6487 0.0097 0.6612 0.0102 0.5660 0.0185  0.2303    42.3000
    llama3.1_8b  Standard 0.6320 0.0056 0.6469 0.0121 0.5643 0.0161  0.5563    57.3000

True class distribution (Experiment B):
true_label
credit_reporting      51.1000
debt_collection       18.2000
mortgages_and_loans   10.7000
credit_card           10.0000
retail_bank

---
## 4. Per-Class F1 — Balanced N=500 (averaged over 3 seeds)

In [5]:
rows = []
for _, row in agg_a.iterrows():
    r = {"Model": row["model"], "Type": row["type"]}
    for cat in CATEGORIES:
        r[cat] = round(row[f"f1_{cat}_mean"], 4)
    rows.append(r)
pc = pd.DataFrame(rows)
pc.columns = ["Model","Type","Retail Bkg","Credit Rep.","Credit Card","Mortg. & Loans","Debt Coll."]
print("Per-Class F1 — Balanced N=500 (mean over 3 seeds)")
print("=" * 85)
print(pc.to_string(index=False))

Per-Class F1 — Balanced N=500 (mean over 3 seeds)
          Model      Type  Retail Bkg  Credit Rep.  Credit Card  Mortg. & Loans  Debt Coll.
    gpt-oss_20b Reasoning      0.7939       0.7137       0.7394          0.7953      0.7045
deepseek-r1_14b Reasoning      0.7830       0.6968       0.7101          0.7535      0.6764
       qwen3_8b Reasoning      0.7802       0.6773       0.7156          0.7694      0.6437
       phi4_14b  Standard      0.7108       0.7057       0.7041          0.7370      0.6702
 deepseek-r1_8b Reasoning      0.7480       0.6602       0.6695          0.7574      0.5793
      gemma2_9b  Standard      0.6753       0.6911       0.6614          0.7851      0.5685
     mistral_7b  Standard      0.2959       0.6907       0.6179          0.7114      0.6182
    llama3.1_8b  Standard      0.6451       0.6481       0.6218          0.5909      0.4810


---
## 5. Sample Size Sensitivity (N=250 vs N=500)

In [6]:
rows = []
for model in MODELS:
    for size in [250, 500]:
        for cond in ["balanced", "imbalanced"]:
            sub = raw[(raw["model"]==model) & (raw["condition"]==cond) & (raw["size"]==size)]
            if sub.empty: continue
            rows.append({
                "Model": model,
                "Condition": cond,
                "N": size,
                "Acc mean": round(sub["accuracy"].mean(), 4),
                "Acc std":  round(sub["accuracy"].std(ddof=1) if len(sub)>1 else 0, 4),
                "F1w mean": round(sub["f1_weighted"].mean(), 4),
                "F1w std":  round(sub["f1_weighted"].std(ddof=1) if len(sub)>1 else 0, 4),
            })
size_df = pd.DataFrame(rows)
print("Sample Size Sensitivity — Balanced")
print("=" * 65)
print(size_df[size_df["Condition"]=="balanced"].drop(columns="Condition").to_string(index=False))
print("\nSample Size Sensitivity — Imbalanced")
print("=" * 65)
print(size_df[size_df["Condition"]=="imbalanced"].drop(columns="Condition").to_string(index=False))

Sample Size Sensitivity — Balanced
          Model   N  Acc mean  Acc std  F1w mean  F1w std
 deepseek-r1_8b 250    0.6871   0.0152    0.6843   0.0160
 deepseek-r1_8b 500    0.6843   0.0129    0.6829   0.0119
deepseek-r1_14b 250    0.7082   0.0150    0.7077   0.0146
deepseek-r1_14b 500    0.7237   0.0064    0.7240   0.0068
      gemma2_9b 250    0.6773   0.0032    0.6747   0.0049
      gemma2_9b 500    0.6797   0.0064    0.6763   0.0085
    gpt-oss_20b 250    0.7472   0.0170    0.7477   0.0159
    gpt-oss_20b 500    0.7483   0.0199    0.7494   0.0185
    llama3.1_8b 250    0.5497   0.0055    0.5675   0.0046
    llama3.1_8b 500    0.5896   0.0188    0.5982   0.0178
     mistral_7b 250    0.5853   0.0031    0.5775   0.0038
     mistral_7b 500    0.5933   0.0023    0.5868   0.0039
       phi4_14b 250    0.6979   0.0116    0.6983   0.0112
       phi4_14b 500    0.7050   0.0115    0.7056   0.0107
       qwen3_8b 250    0.7102   0.0057    0.7100   0.0052
       qwen3_8b 500    0.7167   0.003

---
## 6. McNemar Tests — Experiment A (per seed, then pooled)

In [7]:
def correct_vec(df):
    return (df["true_label"] == df["predicted_label"]).astype(int).values

def exact_mcnemar(v1, v2):
    b = int(((v1==1)&(v2==0)).sum())
    c = int(((v1==0)&(v2==1)).sum())
    n = b + c
    if n == 0: return b, c, 1.0
    p = min(2 * binom.cdf(min(b,c), n, 0.5), 1.0)
    return b, c, p

ALPHA = 0.05 / 4  # Bonferroni, 4 comparisons vs best model
best = "gpt-oss_20b"
compare_to = ["deepseek-r1_14b", "phi4_14b", "qwen3_8b", "gemma2_9b"]

print(f"Bonferroni alpha (4 comparisons): {ALPHA:.4f}\n")
print(f"{'Seed':>4}  {'vs Model':22s}  {'b':>4}  {'c':>4}  {'p-value':>10}  sig?")
print("-" * 60)

for seed in [1, 2, 3]:
    sub = raw[(raw["condition"]=="balanced") & (raw["size"]==500) & (raw["seed"]==seed)]
    v_best = correct_vec(sub[sub["model"]==best]["_df"].values[0])
    for other in compare_to:
        v_other = correct_vec(sub[sub["model"]==other]["_df"].values[0])
        b, c, p = exact_mcnemar(v_best, v_other)
        sig = "YES" if p < ALPHA else "no"
        print(f"  {seed:2d}  {other:22s}  {b:4d}  {c:4d}  {p:10.4f}  {sig}")
    print()

Bonferroni alpha (4 comparisons): 0.0125

Seed  vs Model                   b     c     p-value  sig?
------------------------------------------------------------
   1  deepseek-r1_14b           75    64      0.3964  no
   1  phi4_14b                  96    66      0.0224  no
   1  qwen3_8b                  76    58      0.1417  no
   1  gemma2_9b                118    64      0.0001  YES

   2  deepseek-r1_14b           88    49      0.0011  YES
   2  phi4_14b                 110    58      0.0001  YES
   2  qwen3_8b                  91    39      0.0000  YES
   2  gemma2_9b                139    44      0.0000  YES

   3  deepseek-r1_14b           72    48      0.0353  no
   3  phi4_14b                  92    44      0.0000  YES
   3  qwen3_8b                  77    52      0.0342  no
   3  gemma2_9b                 96    39      0.0000  YES



---
## 7. Performance vs. Commercial API — Cost-Accuracy Table

In [8]:
ref_rows = [{"Model": k, "Source": "Commercial API", "Acc": v["acc"],
             "Acc±": "—", "F1w": v["f1"], "F1w±": "—", "Cost/1k": v["cost"]}
            for k, v in REFERENCE.items()]
our_rows = [{"Model": row["model"], "Source": "Open-source (ours)",
             "Acc":  round(row["accuracy_mean"],    4),
             "Acc±": round(row["accuracy_std"],     4),
             "F1w":  round(row["f1_weighted_mean"], 4),
             "F1w±": round(row["f1_weighted_std"],  4),
             "Cost/1k": "$0.00"}
            for _, row in agg_a.iterrows()]

combined = pd.DataFrame(ref_rows + our_rows)
combined["_sort"] = pd.to_numeric(combined["Acc"], errors="coerce")
combined = combined.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)

print("Zero-Shot Performance and Cost: Open-Source vs. Commercial Models (Balanced N=500)")
print("=" * 85)
print(combined.to_string(index=False))

best_f1   = agg_a.loc[agg_a["model"]=="gpt-oss_20b", "f1_weighted_mean"].values[0]
best_std  = agg_a.loc[agg_a["model"]=="gpt-oss_20b", "f1_weighted_std"].values[0]
gpt4o_f1  = 0.7358
print(f"\ngpt-oss:20b F1w = {best_f1:.4f} ± {best_std:.4f}  |  GPT-4o F1w = {gpt4o_f1:.4f}")
print(f"gpt-oss:20b > GPT-4o: {best_f1 > gpt4o_f1}  (lower bound {best_f1-best_std:.4f} vs GPT-4o {gpt4o_f1})")

Zero-Shot Performance and Cost: Open-Source vs. Commercial Models (Balanced N=500)
                       Model             Source    Acc   Acc±    F1w   F1w± Cost/1k
               o1 (paid API)     Commercial API 0.7770      — 0.7764      —  $24.24
          GPT-4.5 (paid API)     Commercial API 0.7740      — 0.7763      —  $12.32
Claude 3.7 Sonnet (paid API)     Commercial API 0.7650      — 0.7626      —   $0.64
 Gemini 2.0 Flash (paid API)     Commercial API 0.7580      — 0.7586      —   $0.04
                 gpt-oss_20b Open-source (ours) 0.7483 0.0199 0.7494 0.0185   $0.00
           GPT-4o (paid API)     Commercial API 0.7360      — 0.7358      —   $0.45
             deepseek-r1_14b Open-source (ours) 0.7237 0.0064 0.7240 0.0068   $0.00
                    qwen3_8b Open-source (ours) 0.7167 0.0032 0.7172 0.0023   $0.00
                    phi4_14b Open-source (ours) 0.7050 0.0115 0.7056 0.0107   $0.00
              deepseek-r1_8b Open-source (ours) 0.6843 0.0129 0.6829 0.0119  

---
## 8. Off-Label Counts (mean over 3 seeds)

In [9]:
print(f"{'Model':22s}  {'Type':10s}  {'Exp A OL (mean)':>15}  {'Exp B OL (mean)':>15}")
print("-" * 68)
for model in MODELS:
    mtype = "Reasoning" if model in REASONING else "Standard"
    ol_a = raw[(raw["model"]==model)&(raw["condition"]=="balanced")&(raw["size"]==500)]["off_label"].mean()
    ol_b = raw[(raw["model"]==model)&(raw["condition"]=="imbalanced")&(raw["size"]==500)]["off_label"].mean()
    print(f"{model:22s}  {mtype:10s}  {ol_a:15.1f}  {ol_b:15.1f}")

Model                   Type        Exp A OL (mean)  Exp B OL (mean)
--------------------------------------------------------------------
deepseek-r1_8b          Reasoning               2.0              2.3
deepseek-r1_14b         Reasoning               1.3              0.3
gemma2_9b               Standard                0.3              0.3
gpt-oss_20b             Reasoning               0.0              0.0
llama3.1_8b             Standard               56.0             57.3
mistral_7b              Standard               58.7             42.3
phi4_14b                Standard                1.0              0.0
qwen3_8b                Reasoning               0.0              0.3


---
## 9. Reproducibility Notes

| Item | Value |
|------|-------|
| Prediction CSV dir | `preds/` |
| File naming | `preds_{model}_{seed}_subset_{condition}_{size}.csv` |
| Seeds | 1, 2, 3 |
| Sample sizes | 250, 500 |
| Inference script | `run_inference.py` |
| Analysis script | `analyze_seeds.py` |
| SLURM job file | `slurm_inference_array.sh` |
| GPU | NVIDIA Tesla V100-SXM2-16GB |
| Cluster | TRUBA ARF, akya-cuda partition |
| Temperature | 0 (all models) |
| Quantisation | Q4_K_M (Ollama default) |
| Timeout per request | 300 s |